# Automated Data Filler Agent -- Demo Notebook

Run this notebook from the `notebooks/` folder. It imports the `agent` package from the project root, so the first cell fixes the path.

## 0. Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))  # so `import agent...` works from notebooks/

# This notebook uses the LOCAL Ollama model -- no API key needed.
# Before running: open the Ollama app, and make sure you've run
# `ollama pull llama3.1` in a terminal at least once.
import requests
try:
    requests.get("http://localhost:11434", timeout=3)
    print("Ollama is running -- good to go.")
except requests.exceptions.ConnectionError:
    print("WARNING: Ollama doesn't seem to be running. Open the Ollama app first.")

## 1. Extract CRM data from a single meeting summary

In [ ]:
from agent.extractor_ollama import extract_crm_data
from agent.validator import validate_extraction, ValidationError
import json

summary = (
    "Had a call with Sarah Johnson, Marketing Director at GrowthTech. "
    "They need marketing automation for 50-person team. Budget: $30K annually. "
    "Evaluating HubSpot vs our solution. Next: Demo scheduled for Friday."
)

result = extract_crm_data(summary)
print(json.dumps(result, indent=2))

In [ ]:
# Validate before it would go anywhere near a real CRM
try:
    warnings = validate_extraction(result)
    print("Valid. Warnings:", warnings if warnings else "none")
except ValidationError as e:
    print("Validation failed:", e)

## 2. Try it on your own text
Paste any meeting summary below and re-run.

In [ ]:
my_summary = """PASTE YOUR OWN MEETING SUMMARY HERE"""

result2 = extract_crm_data(my_summary)
print(json.dumps(result2, indent=2))

## 3. Quick evaluation against a few test examples
Loads a handful of labeled rows from `data/merged_test.jsonl` (source == synthetic, which have full ground truth) and compares extraction output field-by-field.

In [ ]:
import json

with open("../data/merged_test.jsonl") as f:
    test_rows = [json.loads(l) for l in f]

labeled_test_rows = [r for r in test_rows if r["source"] == "synthetic"][:5]

for row in labeled_test_rows:
    pred = extract_crm_data(row["meeting_summary"])
    truth = row["salesforce_record"]
    print("SUMMARY:", row["meeting_summary"][:100], "...")
    print("  Predicted Contact Name:", pred["Contact"]["FirstName"], pred["Contact"]["LastName"],
          "| Actual:", truth["Contact"]["FirstName"], truth["Contact"]["LastName"])
    print("  Predicted Company:", pred["Account"]["Name"], "| Actual:", truth["Account"]["Name"])
    print("  Predicted Amount:", pred["Opportunity"]["Amount"], "| Actual:", truth["Opportunity"]["Amount"])
    print()